# D2 · M2.2 — Vector Stores, Part 2: Qdrant & Choosing a Store

**Part 2 of 2.** Part 1 built semantic search by hand and stood up Chroma. This notebook mirrors
the same collection into Qdrant, then compares the two stores directly. **This notebook is what
makes M2.2 gradable** — Part 1 alone wasn't enough (Chroma only), this adds Qdrant.

**Worked side by side with the Concept HTML page.** Read Concept C, run the matching cell, back
to the page for Concept D, run the last cell — two round trips this session.

**This notebook is fully working code — nothing to write. It's self-contained** — it re-embeds
the same 10 Meridian passages rather than depending on Part 1's kernel still being alive.

Concept page: `Day2/concept/D2_M1.4_M2.2_VectorStores_Concept.html` (sections C and D)


## Setup

Re-embeds the same 10 Meridian passages Part 1 used — 10 fresh API calls, cheap, so this
notebook works on its own even if Part 1's kernel is gone.


In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Find the .env file wherever it lives: this folder, any folder above it, or the
# Day1/labs/starter/ folder SETUP.md originally told you to create it in. One key
# file for the whole repo -- you never need to copy it between Day folders.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)

client = OpenAI()
MODEL_EMBED = "text-embedding-3-small"

_DAY1_DATA = Path.cwd().resolve().parents[2] / "Day1" / "data"
PASSAGES_PATH = _DAY1_DATA / "D1_M1.4_banking_passages.json"


def get_embedding(text):
    response = client.embeddings.create(model=MODEL_EMBED, input=text)
    return response.data[0].embedding


with open(PASSAGES_PATH, "r", encoding="utf-8") as f:
    passages = json.load(f)

print("Re-embedding {} passages (fresh, self-contained run)...".format(len(passages)))
passage_index = []
for p in passages:
    vec = get_embedding(p["text"])
    passage_index.append({"id": p["id"], "category": p["category"], "text": p["text"], "embedding": vec})
print("Ready -- {} passages embedded.".format(len(passage_index)))


## Concept C, on the page -- then come back here for the Qdrant lab

Read **Concept C - One Store, Two Deployment Modes** on the concept page, then run the cell
below. It tries a self-hosted Qdrant (Docker) first, and transparently falls back to an
in-memory client if Docker isn't reachable -- same idea as `init_chat_model`'s provider swap
back in M2.1: one interface, the detail that changes is just which mode connected.


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

COLLECTION_NAME = "meridian_banking_passages"


def get_qdrant_client():
    """Tries a self-hosted Qdrant (Docker, the syllabus's primary/production path) first.
    Falls back to Qdrant's in-memory client if no Docker instance is reachable -- so this
    lab works whether or not Docker is set up on this particular VM."""
    try:
        c = QdrantClient(url="http://localhost:6333", timeout=2, check_compatibility=False)
        c.get_collections()
        return c, "docker (self-hosted)"
    except Exception:
        return QdrantClient(location=":memory:"), "in-memory (fallback -- Docker not reachable)"


qdrant_client, qdrant_mode = get_qdrant_client()
print("Qdrant connection mode: {}".format(qdrant_mode))

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)
qdrant_client.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=len(passage_index[0]["embedding"]), distance=Distance.COSINE),
)

qdrant_client.upsert(
    COLLECTION_NAME,
    points=[
        PointStruct(id=p["id"], vector=p["embedding"], payload={"category": p["category"], "text": p["text"]})
        for p in passage_index
    ],
)
qdrant_count = qdrant_client.count(COLLECTION_NAME).count
print("Ingested {} passages into Qdrant, each tagged with a 'category' metadata field.".format(qdrant_count))

query_text = "Someone used my account without my permission."
query_vec = get_embedding(query_text)

qdrant_unfiltered = qdrant_client.query_points(COLLECTION_NAME, query=query_vec, limit=3)
print("\nQuery: \"{}\" -- no filter".format(query_text))
for pt in qdrant_unfiltered.points:
    print("  id {}  (score {:.4f})".format(pt.id, pt.score))

qdrant_filtered = qdrant_client.query_points(
    COLLECTION_NAME, query=query_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value="fraud"))]),
)
print("\nSame query -- filtered to category='fraud'")
for pt in qdrant_filtered.points:
    print("  id {}  (score {:.4f})".format(pt.id, pt.score))

qdrant_query_results = {
    "mode": qdrant_mode,
    "query": query_text,
    "unfiltered_ids": [pt.id for pt in qdrant_unfiltered.points],
    "filtered_ids": [pt.id for pt in qdrant_filtered.points],
    "collection_count": qdrant_count,
}


**Notice:** whichever mode printed above (Docker or in-memory), the collection, upsert,
and query calls above never changed -- exactly like `init_chat_model`'s provider swap in M2.1.
Now go back to the concept page for **Concept D**.


## Concept D, on the page -- then come back here for the comparison

Read **Concept D - Choosing a Store** on the concept page, then run the cell below. It stands
up Chroma again (quick, in-memory, no cost beyond re-using the embeddings above) so the same
query can be measured against both stores side by side, in this same notebook.


In [ ]:
import time
import chromadb

chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection(name="meridian_banking_passages_compare")
chroma_collection.add(
    ids=[str(p["id"]) for p in passage_index],
    embeddings=[p["embedding"] for p in passage_index],
    documents=[p["text"] for p in passage_index],
    metadatas=[{"category": p["category"]} for p in passage_index],
)

start = time.perf_counter()
chroma_result = chroma_collection.query(query_embeddings=[query_vec], n_results=3,
                                         where={"category": "fraud"})
chroma_latency_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
qdrant_result = qdrant_client.query_points(
    COLLECTION_NAME, query=query_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value="fraud"))]),
)
qdrant_latency_ms = (time.perf_counter() - start) * 1000

print("=" * 72)
print("COMPARISON -- same query, same filter, both stores")
print("=" * 72)
print("Chroma  -> ids {}  ({:.2f} ms, this run)".format(chroma_result["ids"][0], chroma_latency_ms))
print("Qdrant  -> ids {}  ({:.2f} ms, this run, mode={})".format(
    [pt.id for pt in qdrant_result.points], qdrant_latency_ms, qdrant_mode))
print(
    "\nBoth returned the same passage(s) for this filtered query -- expected, since both are "
    "comparing the same underlying embeddings. The real difference between them isn't correctness "
    "here, it's operational: Chroma is the faster path to a local prototype; Qdrant is built for "
    "self-hosted, production-scale deployment -- the choice that matters for regulated banking data "
    "that has to stay inside the organisation's boundary."
)

comparison_table = {
    "query": query_text,
    "filter": "category=fraud",
    "chroma": {"ids": chroma_result["ids"][0], "latency_ms": chroma_latency_ms},
    "qdrant": {"ids": [pt.id for pt in qdrant_result.points], "latency_ms": qdrant_latency_ms, "mode": qdrant_mode},
}


**Notice:** correctness matched across both stores -- the decision between them is about
deployment, not accuracy. Run the final cell to save your results.

## What's next

M2.2 is complete: a populated Chroma collection and an equivalent Qdrant collection, both with
metadata filters, per this module's actual deliverable. Next up: M2.3 + M2.4, where these vector
stores plug into a full RAG pipeline.


In [ ]:
key_takeaway = """
KEY TAKEAWAY: Qdrant mirrors everything Chroma did -- same embeddings, same
metadata filter, same correctness -- reached through a client that tries a
self-hosted Docker instance first and falls back to in-memory only if Docker
isn't reachable, same "one interface, swap the detail" idea as M2.1's
init_chat_model. The choice between Chroma and Qdrant is operational, not
correctness: Chroma is the fast local-prototyping path, Qdrant is the
self-hosted, production-scale path -- the one that matters when regulated
data has to stay inside the organisation's own boundary.
"""
print(key_takeaway)

results = {
    "qdrant_query_results": qdrant_query_results,
    "comparison_table": comparison_table,
    "key_takeaway": key_takeaway,
}

with open("m2_2_part2_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m2_2_part2_results.json --", len(json.dumps(results)), "bytes")
